# Hysteresis Processing

## Install and import packages

In [1]:
import pmagpy.rockmag as rmag
import pmagpy.ipmag as ipmag
import pmagpy.contribution_builder as cb
import pandas as pd
import numpy as np

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

## Import local data in MagIC format
In this demonstration we will be using local data.

The data is from the following publication:
- Swanson-Hysell, N. L., Avery, M. S., Zhang, Y., Hodgin, E. B., Sherwood, R. J., Apen, F. E., et al. (2021). The paleogeography of Laurentia in its early years: New constraints from the Paleoproterozoic East-Central Minnesota Batholith. Tectonics, 40, e2021TC006751. https://doi.org/10.1029/2021TC006751

In [2]:
# set the dir_path to the directory where the measurements.txt file is located
dir_path = '../example_data/ECMB'

# set the name of the MagIC file
ipmag.unpack_magic('magic_contribution_20213.txt', 
                     dir_path = dir_path,
                     input_dir_path = dir_path,
                     print_progress=False)

# create a contribution object from the tables in the directory
contribution = cb.Contribution(dir_path)
measurements = contribution.tables['measurements'].df
specimens = contribution.tables['specimens'].df

1  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/contribution.txt
1  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/locations.txt
90  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/sites.txt
312  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/samples.txt
1574  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/specimens.txt


17428  records written to file  /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/measurements.txt


-I- Using online data model


-I- Getting method codes from earthref.org


-I- Importing controlled vocabularies from https://earthref.org


## Get hysteresis data using the method code

The method codes relevant to hysteresis loops are:
 - `LP-HYS` for regular hysteresis loops
 - `LP-HYS-O` for hysteresis loops as a function of orientation
 - `LP-HYS-T` for hysteresis loops as a function of temperature

We can filter the data to only have data for `LP-HYS`.

In [3]:
hyst_measurements = measurements[measurements['method_codes'] == 'LP-HYS']

We can see what experimental hysteresis data are available in this dataset:

In [4]:
hyst_experiments = rmag.make_experiment_df(hyst_measurements)
hyst_experiments

,specimen,method_codes,experiment
0,NED1-5c,LP-HYS,IRM-VSM3-LP-HYS-218845
1,NED18-2c,LP-HYS,IRM-VSM3-LP-HYS-218847
2,NED2-8c,LP-HYS,IRM-VSM3-LP-HYS-218849
3,NED4-1c,LP-HYS,IRM-VSM3-LP-HYS-218853
4,NED6-6c,LP-HYS,IRM-VSM3-LP-HYS-218855


## Process hysteresis data for one experiment

We can process the data from one of the experiments using the `rmag.process_hyst_loop` function. The processing steps behind the function are described in the [hysteresis_processing_walk_through.ipynb](./hysteresis_processing_walk_through.ipynb) notebook.

In [5]:
#specify the experiment name and specimen name
experiment_name = 'IRM-VSM3-LP-HYS-218845'
specimen_name = 'NED1-5c'

experiment_hyst = measurements[measurements['experiment'] == experiment_name].reset_index(drop=True)
experiment_results = rmag.process_hyst_loop(experiment_hyst['meas_field_dc'].values, experiment_hyst['magn_mass'].values, specimen_name)
NED1_5c_hyst_process_result = experiment_results

## Process hysteresis data for all experiments

In [6]:
results = rmag.process_hyst_loops(hyst_experiments,measurements)

## Export hysteresis loop statistics into a MagIC specimens data table

The `add_hyst_stats_to_specimens_table` function adds the summary hysteresis parameters (`hyst_ms_mass`, `hyst_mr_mass`, `hyst_bc`, `hyst_xhf`) to the MagIC columns of the specimens table, with the additional quality statistics (Q, Qf, FNL values, sigma, etc.) recorded in the `description` column.

In [7]:
specimens = rmag.add_hyst_stats_to_specimens_table(specimens, results)

In [8]:
specimens

,citations,critical_temp,critical_temp_type,description,dir_comp,dir_dang,dir_dec,dir_inc,dir_mad_free,dir_n_comps,...,rem_hirm_mass,rem_sratio,rem_sratio_back,rem_sratio_forward,result_quality,sample,software_packages,specimen,volume,weight
specimen name,,,,,,,,,,,,,,,,,,,,,
NED1-1b,This study,NaN,NaN,NaN,mc,2.8,159.7,62.5,1.0,1.0,...,NaN,NaN,NaN,NaN,g,NED1-1,pmagpy-4.2.25: demag_gui,NED1-1b,NaN,NaN
NED1-1b,This study,NaN,NaN,NaN,mc,2.8,139.2,74.1,1.0,1.0,...,NaN,NaN,NaN,NaN,g,NED1-1,pmagpy-4.2.25: demag_gui,NED1-1b,NaN,NaN
NED1-1b,This study,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NED1-1,NaN,NED1-1b,NaN,NaN
NED1-2b,This study,NaN,NaN,NaN,mc,3.4,164.1,66.9,0.4,1.0,...,NaN,NaN,NaN,NaN,g,NED1-2,pmagpy-4.2.25: demag_gui,NED1-2b,NaN,NaN
NED1-2b,This study,NaN,NaN,NaN,mc,3.4,149.9,76.3,0.4,1.0,...,NaN,NaN,NaN,NaN,g,NED1-2,pmagpy-4.2.25: demag_gui,NED1-2b,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NED1-5c,This study,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.000007,0.931,0.1,1.0,NaN,NED1-5,NaN,NED1-5c,NaN,0.000154
NED18-2c,This study,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000066,0.938,0.1,1.0,NaN,NED18-2,NaN,NED18-2c,NaN,0.000254
NED2-8c,This study,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000074,0.846,0.1,1.0,NaN,NED2-8,NaN,NED2-8c,NaN,0.000188


The updated table can then be written out as a MagIC-format `specimens.txt` file:

In [9]:
contribution.tables['specimens'].df = specimens
contribution.tables['specimens'].write_magic_file(dir_path=dir_path)

-I- overwriting /Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/specimens.txt
-I- 1574 records written to specimens file


'/Users/penokean/0000_GitHub/RockmagPy-notebooks/example_data/ECMB/specimens.txt'

## Processing hysteresis data that are not in MagIC format

While the workflow above is built around MagIC-formatted data — the recommended practice, since it keeps measurement-level data, metadata, and derived parameters together — the processing functions themselves operate on simple sequences of field and magnetization values. This makes these algorithms accessible to data coming from any stream: an instrument export, a spreadsheet, or a colleague's text file.

A few conventions to be aware of:
- **Field values are expected in tesla.** The high-field susceptibility unit conversions assume tesla, and a warning is printed if the field values appear to be in mT or Oe.
- **Magnetization can be in any consistent unit.** Mass-normalized Am²/kg matches MagIC conventions and makes the resulting M<sub>s</sub> and M<sub>r</sub> directly comparable to database values.
- Inputs are cleaned automatically by `sanitize_hysteresis_inputs`: plain Python lists, numeric strings, and dataframe columns are all accepted; measurement pairs with non-finite values are dropped with a report; and loops measured in either field sweep order (starting from positive or negative saturation) are handled.

Below we process a plain two-column CSV file (the NED18-2c loop exported with `field_tesla` and `moment_Am2_per_kg` columns):

In [10]:
plain_data = pd.read_csv('../example_data/plain_text/NED18_2c_hysteresis.csv')
plain_result = rmag.process_hyst_loop(plain_data['field_tesla'],
                                      plain_data['moment_Am2_per_kg'],
                                      'NED18-2c (from csv)')